## muMAG Standard Problem #4 

put some description here

Install magnum.np and import:

In [2]:
!pip install -q matplotlib==3.6.2
!pip install -q magnumnp

from magnumnp import *

initialize mesh and setup materials:

In [3]:
Timer.enable(log_mem = True)

# initialize mesh
eps = 1e-15
n  = (100, 25, 1)
dx = (5e-9, 5e-9, 3e-9)
mesh = Mesh(n, dx)
state = State(mesh)

state.material = {
    "Ms": 8e5,
    "A": 1.3e-11,
    "alpha": 0.02
    }

/home/florian/.local/lib/python3.8/site-packages/torch/cuda/__init__.py:82: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 10010). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at  ../c10/cuda/CUDAFunctions.cpp:112.)
  return torch._C._cuda_getDeviceCount() > 0
2023-02-13 16:30:06  magnum.np:INFO [State] running on device: cpu (dtype = float32)
2023-02-13 16:30:06  magnum.np:INFO [Mesh] 100x25x1 (size= 5e-09 x 5e-09 x 3e-09)


some more text.
initialize field terms and magnetization:

In [4]:
# initialize field terms
demag    = DemagField()
exchange = ExchangeField()
external = ExternalField([-24.6e-3/constants.mu_0,
                          +4.3e-3/constants.mu_0,
                          0.0])

# initialize magnetization that relaxes into s-state
state.m = state.Constant([0,0,0])
state.m[1:-1,:,:,0]   = 1.0
state.m[(-1,0),:,:,1] = 1.0

more text.
Iterate in Time using the default RKF45 solver:

In [5]:
# relax without external field
llg = LLGSolver([demag, exchange])
llg.relax(state)
write_vti(state.m, "data/m0.vti", state)

# perform integration with external field
llg = LLGSolver([demag, exchange, external], solver = ScipyODE)
logger = Logger("data", ['t', 'm'])
while state.t < 1e-9-eps:
    llg.step(state, 1e-11)
    logger << state

Timer.print_report()

2023-02-13 15:27:35  magnum.np:INFO [LLGSolver] using RKF45 solver (atol = 1e-05)
INFO:magnum.np:[LLGSolver] using RKF45 solver (atol = 1e-05)
2023-02-13 15:27:37  magnum.np:INFO [DEMAG]: Time calculation of demag kernel = 1.2837371826171875 s
INFO:magnum.np:[DEMAG]: Time calculation of demag kernel = 1.2837371826171875 s
2023-02-13 15:27:37  magnum.np:INFO [LLG] relax: t=1e-11 dE=3.992 E=9.25329e-19
INFO:magnum.np:[LLG] relax: t=1e-11 dE=3.992 E=9.25329e-19
2023-02-13 15:27:37  magnum.np:INFO [LLG] relax: t=2e-11 dE=0.267094 E=7.30277e-19
INFO:magnum.np:[LLG] relax: t=2e-11 dE=0.267094 E=7.30277e-19
2023-02-13 15:27:37  magnum.np:INFO [LLG] relax: t=3e-11 dE=0.0521441 E=6.94084e-19
INFO:magnum.np:[LLG] relax: t=3e-11 dE=0.0521441 E=6.94084e-19
2023-02-13 15:27:37  magnum.np:INFO [LLG] relax: t=4e-11 dE=0.0178207 E=6.81932e-19
INFO:magnum.np:[LLG] relax: t=4e-11 dE=0.0178207 E=6.81932e-19
2023-02-13 15:27:37  magnum.np:INFO [LLG] relax: t=5e-11 dE=0.00873075 E=6.76029e-19
INFO:magnum.n


TIMER REPORT
Operation              No of calls    Avg time [ms]    Total time [s]    Memory [MB]
-------------------  -------------  ---------------  ----------------  -------------
LLGSolver.relax                  1    17820.8               17.8208        12.1094
    DemagField.h              3882        2.2958             8.9123        11.4336
    ExchangeField.h           3882        1.45916            5.66444        0
LLGSolver.step                 100      239.015             23.9015         3.69531
    DemagField.h              4947        2.07288           10.2546         1.55078
    ExchangeField.h           4947        1.60869            7.95819        0.175781
    ExternalField.h           4947        0.0887603          0.439097       0
-------------------  -------------  ---------------  ----------------  -------------
Total                                                      112.373
Missing                                                     70.6504

